# U-Net para Segmentação Semântica — Oxford-IIIT Pet

**Disciplina:** BCC6003 – Inteligência Computacional  
**Tópico:** Segmentação binária (animal vs. fundo) com fine-tuning de U-Net

## Objetivos

Ao concluir este notebook, você deve ser capaz de:

- compreender a estrutura encoder–decoder da U-Net e o papel das **skip connections** (visto na aula);
- configurar uma U-Net com encoder pré-treinado via `segmentation_models.pytorch`;
- treinar uma rede de segmentação com `BCEWithLogitsLoss`;
- visualizar e discutir máscaras previstas.

**Referências:**
- Ronneberger, O.; Fischer, P.; Brox, T. *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI, 2015.
- [segmentation_models.pytorch](https://github.com/qubvel-org/segmentation_models.pytorch)

---

## Configuração no Kaggle (obrigatório)

Antes de executar, ajuste as configurações do notebook:

1. **Settings → Accelerator → GPU T4** (ou GPU disponível)
2. **Settings → Internet → On** — necessário para baixar o dataset, a biblioteca e os pesos ImageNet
3. Após concluir, use **Save Version → Save & Run All** antes de compartilhar ou incorporar o notebook

## Configuração no Google Colab

1. **Ambiente de execução → Alterar tipo de ambiente → T4 GPU**
2. Internet ativa (padrão no Colab)

> **Dica:** o treinamento com 5 épocas deve levar cerca de 3–5 minutos na GPU T4 (encoder ResNet18 pré-treinado).


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import functional as TF

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
%pip install -q segmentation-models-pytorch

import segmentation_models_pytorch as smp

print(f"segmentation_models.pytorch — U-Net disponível: {hasattr(smp, 'Unet')}")


## 1. Carregamento do dataset

Utilizamos o **Oxford-IIIT Pet Dataset** via `torchvision.datasets.OxfordIIITPet`.

- **Imagens:** fotos de gatos e cachorros
- **Máscaras (trimap):** pixel `1` = animal, `2` = fundo, `3` = contorno

Para segmentação **binária**, tratamos apenas pixels do animal (`1`) como foreground; contorno e fundo viram `0`.


In [ ]:
DATA_ROOT = "/kaggle/working/data"
IMG_SIZE = (128, 128)

# Normalização ImageNet (compatível com encoder pré-treinado)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class PetSegmentationDataset(torch.utils.data.Dataset):
    """Wrapper com resize, tensor e máscara binária."""

    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, trimap = self.base[idx]
        image = TF.resize(image, IMG_SIZE)
        trimap = TF.resize(trimap, IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST)

        image = TF.to_tensor(image)
        image = TF.normalize(image, IMAGENET_MEAN, IMAGENET_STD)

        mask = torch.as_tensor(np.array(trimap), dtype=torch.uint8)
        mask = (mask == 1).float().unsqueeze(0)  # (1, H, W)

        return image, mask


base_ds = OxfordIIITPet(
    root=DATA_ROOT,
    split="trainval",
    target_types="segmentation",
    download=True,
)

full_ds = PetSegmentationDataset(base_ds)
print(f"Total de imagens: {len(full_ds)}")

# Amostra reduzida para treino rápido
N_TRAIN, N_VAL, N_TEST = 600, 100, 100
indices = torch.randperm(len(full_ds), generator=torch.Generator().manual_seed(SEED)).tolist()

train_idx = indices[:N_TRAIN]
val_idx = indices[N_TRAIN : N_TRAIN + N_VAL]
test_idx = indices[N_TRAIN + N_VAL : N_TRAIN + N_VAL + N_TEST]

train_ds = Subset(full_ds, train_idx)
val_ds = Subset(full_ds, val_idx)
test_ds = Subset(full_ds, test_idx)

print(f"Treino: {len(train_ds)} | Validação: {len(val_ds)} | Teste: {len(test_ds)}")


## 2. Pré-processamento

As imagens são redimensionadas para **128×128** e convertidas em tensores normalizados com estatísticas ImageNet.

### Perguntas para reflexão

1. Por que reduzir o tamanho das imagens acelera o treinamento?
2. O que acontece com o consumo de memória quando usamos imagens maiores (ex.: 512×512)?


In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2

pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

# Visualização rápida de um par imagem/máscara
images, masks = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 5))
for i in range(4):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    axes[0, i].imshow(img)
    axes[0, i].axis("off")
    axes[1, i].imshow(masks[i, 0].numpy(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Imagem")
axes[1, 0].set_ylabel("Máscara")
plt.suptitle("Amostras do conjunto de treino")
plt.tight_layout()
plt.show()


## 3. Configuração da U-Net (`segmentation_models.pytorch`)

Na aula estudamos a U-Net original (Ronneberger et al., 2015): encoder contrativo, bottleneck, decoder expansivo e **skip connections** entre camadas de mesma resolução.

Aqui usamos a biblioteca **segmentation_models.pytorch**, que implementa a mesma estrutura com um encoder (ResNet18) **pré-treinado no ImageNet** — técnica de *transfer learning* que acelera a convergência.

### Diagrama de referência (U-Net clássica)

```
Entrada     128×128×3
   │
Encoder     128×128×64 → 64×64×128 → 32×32×256 → 16×16×512
   │
Bottleneck   8×8×1024
   │
Decoder     16×16×512 → 32×32×256 → 64×64×128 → 128×128×64
   │
Saída       128×128×1   (logits — sem Sigmoid na rede)
```

### Perguntas para reflexão

1. O que significa usar um encoder pré-treinado no ImageNet?
2. Por que `activation=None` quando usamos `BCEWithLogitsLoss`?
3. Onde ficam o encoder, o decoder e as skip connections neste modelo?


In [ ]:
# TODO: escolha o encoder (resnet18 é recomendado para treino rápido na T4)
ENCODER_NAME = "resnet18"

# TODO: complete a configuração da U-Net
model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights="imagenet",  # TODO: o que significa este parâmetro?
    in_channels=3,
    classes=1,
    activation=None,  # TODO: por que None com BCEWithLogitsLoss?
).to(device)

print(model)
print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters()):,}")


### Justificativa da escolha do encoder

Escreva abaixo (2–3 linhas) por que você escolheu o encoder indicado (ex.: ResNet18 é leve e converge rápido na GPU T4).


## 4. Função de perda

Usamos **segmentação binária** com `BCEWithLogitsLoss`.

### Perguntas para reflexão

1. Por que usar BCE em segmentação binária?
2. Por que a última camada **não** deve ter Sigmoid quando usamos `BCEWithLogitsLoss`?


In [ ]:
criterion = nn.BCEWithLogitsLoss()
print(criterion)


## 5. Otimizador


In [ ]:
# TODO: configure o otimizador Adam com learning rate 1e-3
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# print(optimizer)


## 6. Loop de treinamento

Treine por **5 épocas** e registre `train_loss` e `val_loss` a cada época.


In [ ]:
NUM_EPOCHS = 5


def train_one_epoch(model, loader, criterion, optimizer, device):
    """TODO: uma época de treino. Retorne a loss média."""
    raise NotImplementedError("Implemente train_one_epoch")


@torch.no_grad()
def validate(model, loader, criterion, device):
    """TODO: validação. Retorne a loss média."""
    raise NotImplementedError("Implemente validate")


history = {"train_loss": [], "val_loss": []}

# TODO: loop de treinamento
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
#     val_loss = validate(model, val_loader, criterion, device)
#     history["train_loss"].append(train_loss)
#     history["val_loss"].append(val_loss)
#     print(f"Época {epoch + 1}/{NUM_EPOCHS} — train: {train_loss:.4f} | val: {val_loss:.4f}")


## 7. Avaliação — curvas de loss

Gere um gráfico comparando perda de treino e validação ao longo das épocas.


In [ ]:
def plot_history(history):
    """TODO: gráfico train_loss vs val_loss por época."""
    raise NotImplementedError("Implemente plot_history")


# plot_history(history)


## 8. Visualização das predições

Selecione imagens do conjunto de **teste** e mostre lado a lado:

**Imagem original** | **Máscara real** | **Máscara predita**


In [ ]:
def denormalize(img_tensor):
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)


def show_predictions(model, dataset, n=4, device=device, threshold=0.5):
    """
    TODO: selecionar n imagens, aplicar sigmoid nos logits,
    threshold 0.5 e plotar 3 colunas (imagem | máscara real | predição).
    """
    raise NotImplementedError("Implemente show_predictions")


# show_predictions(model, test_ds, n=4, device=device)


## 9. Discussão dos resultados

Responda com suas próprias palavras:

1. Em quais imagens a rede acertou? Em quais falhou?
2. O que pode explicar os erros (pose, oclusão, iluminação, contorno)?
3. Mais épocas resolveriam? O que mais poderia melhorar a segmentação?

---

## 10. Questões conceituais (entrega)

1. Explique por que a U-Net utiliza skip connections.
2. Qual é a função do **encoder**?
3. Qual é a função do **decoder**?
4. O que acontece se removermos as skip connections?
5. Por que a última camada possui apenas **um canal**?
6. O que representa cada pixel da saída da U-Net?
7. Por que utilizamos Sigmoid apenas na **inferência** quando treinamos com `BCEWithLogitsLoss`?
8. Cite **duas diferenças** entre uma CNN para classificação e uma U-Net para segmentação semântica.

---

### Desafio extra (opcional)

Substitua `BCEWithLogitsLoss` por **Dice Loss** e compare os resultados visuais.
